In [ ]:
#  conda activate surprise

#  pip install -U sentence-transformers

In [8]:
import surprise
import pandas as pd
import numpy as np


import os
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\ml-100k') 

Semantic Text Embeddings (Sentence Transformers)

In [15]:
from sentence_transformers import SentenceTransformer

In [19]:
 
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
# model = SentenceTransformer("sentence-transformers/all-MiniM-L6-v2")

C:\Users\PGCP-AI.STUDENTSDC\.conda\envs\surprise\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PGCP-AI.STUDENTSDC\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\PGCP-AI.STUDENTSDC\.conda\envs\surprise\lib\site-packages\huggingface_hub\f

In [20]:
sentences = [
    'He batted well today in the cricket match',
    'tommarrow will be wednesday',
    'she played excellently yesterday with many goals',
    'Today will be the event',
    'He performed well in the game'
]

In [21]:
embeddings = model.encode(sentences)
embeddings.shape

(5, 384)

In [22]:
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

torch.Size([5, 5])


In [23]:
similarities

tensor([[1.0000, 0.5786, 0.6758, 0.5816, 0.8370],
        [0.5786, 1.0000, 0.5096, 0.7743, 0.5212],
        [0.6758, 0.5096, 1.0000, 0.5578, 0.7408],
        [0.5816, 0.7743, 0.5578, 1.0000, 0.5403],
        [0.8370, 0.5212, 0.7408, 0.5403, 1.0000]])

# Movie Dataset Processing & Batch Encoding (Netflix Titles)

In [26]:
from tqdm import tqdm
import os 
os.chdir(r'E:\PGCP-AI\PML\Cases\Netflix')

In [41]:
df=  pd.read_csv('titles.csv')
# df

In [32]:
df_indian_movies = df[
    (df['production_countries'].str.contains('IN', na=False)) & 
    (df['type'] == 'MOVIE') & 
    (df['release_year'] >= 1900)
]
df_indian_movies.shape

(577, 15)

In [42]:
# Vectorizing Descriptions:

all_desc = list(df_indian_movies['description'])
out_embs = [] 
for i in tqdm(all_desc):
  enmd = model.encode(i)
  out_embs.append(enmd)
out_embs = np.array(out_embs)
out_embs.shape

100%|██████████| 577/577 [00:22<00:00, 26.03it/s]


(577, 384)

In [37]:
movie_idx = list(df_indian_movies['title']+' - '+ df_indian_movies['release_year'].astype(str))
df_embd = pd.DataFrame(out_embs , index=movie_idx)
df_embd.to_csv('movie-embd.csv')


# Movie search by description

In [48]:
# there are 2 type search  -    keyword search , semantic search
# we are doing semantic search

# query = 'story of college guys'

query = 'techie student in college'

qr_emb = model.encode([query])
sims = []

# Compute similarities  between query embedding and   out embeddings 
for e in tqdm(out_embs):
    sims.append(model.similarity(qr_emb , e).numpy()[0])
    

df_ind = df_indian_movies[['id', 'title', 'description', 'release_year']].copy()
df_ind.loc[: , 'sims'] = np.array(sims)
df_ind.sort_values('sims',ascending=False)[:10]

100%|██████████| 577/577 [00:00<00:00, 6369.81it/s]


,id,title,description,release_year,sims
5391,tm1132325,Don,A youngster who is still figuring out what his...,2022,0.684501
4543,tm838022,Upstarts,"Hoping to do good while making millions, three...",2019,0.591525
492,tm88366,Main Hoon Na,An army major goes undercover as a college stu...,2004,0.589803
1656,tm232947,Brahman Naman,This homage to 1980s teen sex comedies follows...,2016,0.560267
316,tm142564,3 Idiots,Rascal. Joker. Dreamer. Genius... You've never...,2009,0.549468
972,tm185797,Fukrey,The paths of four dream-chasing college friend...,2013,0.539214
4523,tm949905,Asura Guru,"Shakti, who becomes obsessed with money from c...",2020,0.532503
4007,tm858416,Guilty,A young woman from a small town accuses the co...,2020,0.529105
434,tm61056,Dil Chahta Hai,Three inseparable childhood friends are just o...,2001,0.528044
4628,tm885948,"Yours Sincerely, Kanan Gill",Revisiting life goals set in a letter written ...,2020,0.520270


In [ ]:
# find reciprocal rank , mean reciprocal rank